# 03 · Cifras para las 5V
Números del dataset que justifican cada V.
    

In [1]:
from pathlib import Path
import glob
import pandas as pd

LANDING = Path('../data/datalake/landing')
archivos = sorted(glob.glob(str(LANDING / 'usage_events_stream' / '*.jsonl')))
ev = pd.concat([pd.read_json(a, lines=True) for a in archivos], ignore_index=True)
tickets = pd.read_csv(LANDING / 'support_tickets.csv')
print(f'{len(ev):,} eventos')

43,200 eventos


In [2]:
#volumen

bytes_total = sum(Path(a).stat().st_size for a in archivos)
bytes_por_evento = bytes_total / len(ev)
print(f'Muestra: {bytes_total/1e6:.1f} MB · {bytes_por_evento:.0f} bytes por evento · '
      f'{len(ev)/60:.0f} eventos por día')

# SUPUESTO de un proveedor real (ajustable)
RECURSOS, METRICAS, CADA_MIN = 500_000, 3, 5
eventos_dia = RECURSOS * METRICAS * (24 * 60 // CADA_MIN)
print(f'Proyección: {eventos_dia/1e6:.0f} M eventos/día · '
      f'{eventos_dia * bytes_por_evento / 1e9:.0f} GB/día · '
      f'{eventos_dia * bytes_por_evento * 30 / 1e12:.1f} TB/mes')


#con 13 MB alcanza con Excel; con 4 TB por mes no entra en una sola computadora. 
#Eso es lo que justifica Spark, Parquet y un Data Lake. 

Muestra: 12.9 MB · 299 bytes por evento · 720 eventos por día
Proyección: 432 M eventos/día · 129 GB/día · 3.9 TB/mes


In [3]:
#valor (donde esta el costo)
costo = ev['cost_usd_increment']
genai = ev['service'] == 'genai'
print(f'GenAI: {genai.mean():.1%} de los eventos · {costo[genai].sum() / costo.sum():.1%} del costo')
print()
print(costo.groupby(ev['service']).sum().sort_values(ascending=False).round(0))


#genai tiene pocos eventos pero es caro. Cada evento de genai cuesta el doble del promedio. 
#Para FinOps es el servicio a vigilar; para Producto es una señal de pricing.

GenAI: 9.6% de los eventos · 20.0% del costo

service
compute       61787.0
genai         29544.0
database      22635.0
analytics     19709.0
storage        9407.0
networking     4351.0
Name: cost_usd_increment, dtype: float64


In [4]:
#valor para soporte
print(f'SLA incumplido: {tickets.sla_breached.mean():.1%}')
print(f'Tickets abiertos: {tickets.resolved_at.isna().mean():.1%}')
print(f'Tickets high/critical: {tickets.severity.isin(["high", "critical"]).mean():.1%}')

SLA incumplido: 9.5%
Tickets abiertos: 24.0%
Tickets high/critical: 26.0%


## Conclusiones
- **Volumen:** la muestra es chica (12,9 MB, 720 eventos/día). La justificación de Big Data viene de la escala real proyectada (supuesto): ~432 M eventos/día, ~129 GB/día, ~3,9 TB/mes.
- **Valor (FinOps):** costo total ~147 mil USD. Compute es el mayor (61,8 mil). GenAI es el 9,6 % de los eventos pero el 20 % del costo: el servicio más caro por evento.
- **Valor (Soporte):** 9,5 % de tickets con SLA incumplido, 24 % abiertos y 26 % high/critical.